# Introducing Workflows in ADK 2.5.0

In [1]:
import json
from google.adk.workflow import START, Workflow, node
from google.adk.runners import InMemoryRunner

In [3]:
# define the nodes of the workflow


@node
async def calculate_deductions(ctx, node_input: str) -> dict:
    params = json.loads(node_input)
    loan_amount = params["loan_amount"]
    fee_percentage = params["fee_percentage"]
    gst_rate = params["gst_rate"]
    base_fee = loan_amount * (fee_percentage / 100)
    tax = base_fee * (gst_rate / 100)
    total_deductions = base_fee + tax
    print(
        f"[calculate_deductions] base_fee={base_fee}, tax={tax}, total_deductions={total_deductions}"
    )
    return {"loan_amount": loan_amount, "total_deductions": total_deductions}


@node
async def calculate_net_payout(ctx, node_input: dict) -> dict:
    loan_amount = node_input["loan_amount"]
    total_deductions = node_input["total_deductions"]
    net_disbursement = loan_amount - total_deductions
    result = {"net_disbursement": net_disbursement, "status": "READY_FOR_TRANSFER"}
    print(f"[calculate_net_payout] {result}")
    return result

In [4]:
# declare the workflow itself
quick_workflow = Workflow(
    name="quick_workflow",
    edges=[(START, calculate_deductions, calculate_net_payout)],
)

In [7]:
# run the workflow
runner = InMemoryRunner(agent=quick_workflow)
events = await runner.run_debug(
    '{"loan_amount": 50000, "fee_percentage": 2.0, "gst_rate": 18.0}', quiet=True
)
print("Final result:", events[-1].output)

[calculate_deductions] base_fee=1000.0, tax=180.0, total_deductions=1180.0
[calculate_net_payout] {'net_disbursement': 48820.0, 'status': 'READY_FOR_TRANSFER'}
Final result: {'net_disbursement': 48820.0, 'status': 'READY_FOR_TRANSFER'}
